In [34]:
import torch
import torch.nn as nn
import math

In [35]:
VOCAB_SIZE = 50257
BLOCK_SIZE = 256
N_EMBD = 128

N_HEAD = 4
HEAD_SIZE = 32

MLP_HIDEN_DIM = 4
N_TRANSFORMER_BLOCK = 4

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

# Embeddings

In [38]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(block_size, n_embd)


    def forward(self, x):
        size = x.size(dim=-1)
        if (size > self.block_size):
            raise ValueError(f"Input size ({size}) exceded max block size ({self.block_size})")

        idx = torch.arange(size).to(device)

        pos_emb = self.pos_embedding(idx)
        tok_emb = self.tok_embedding(x)

        emb = tok_emb + pos_emb

        return emb


# Self Attention

In [39]:
class SelfAttention(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.head_size = head_size

        self.Q = nn.Linear(n_embd, head_size)
        self.K = nn.Linear(n_embd, head_size)
        self.V = nn.Linear(n_embd, head_size)

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        attention_score = (q @ k.transpose(-1,-2)) / math.sqrt(self.head_size)

        causal_mask = torch.tril(
            torch.ones(
                (attention_score.size(dim=-1), 
                 attention_score.size(dim=-2)), 
                 dtype=torch.bool,
                 device=device
            )
        )

        masked_attention_score = attention_score.masked_fill(~causal_mask, -torch.inf)
        score = torch.softmax(masked_attention_score, dim=-1)

        res = score @ v
        return res

In [40]:
emb = Embedding(VOCAB_SIZE, BLOCK_SIZE, N_EMBD)
x = emb(torch.randint(0, 10, (4, 200)))
print(x.shape)

att = SelfAttention(N_EMBD, HEAD_SIZE)
att(x).shape

torch.Size([4, 200, 128])


torch.Size([4, 200, 32])

# Multi Head Attention

In [41]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        if (n_embd % n_head != 0):
            raise ValueError(f"Number of embeding ({n_embd}) not divisible by number of head ({n_head})")

        self.heads = nn.ModuleList(
            [SelfAttention(n_embd, int(n_embd / n_head)) for _ in range(n_head)]
        )
        self.linear = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        attentions = []

        for head in self.heads:
            attentions.append(head(x))

        attention_cat = torch.cat(attentions, dim=-1)
        res = self.linear(attention_cat)

        return res


In [42]:
emb = Embedding(VOCAB_SIZE, BLOCK_SIZE, N_EMBD)
x = emb(torch.randint(0, 10, (4, 200)))
print(x.shape)
#print(N_EMBD, HEAD_SIZE)
att = MultiHeadAttention(N_EMBD, N_HEAD)
att(x).shape

torch.Size([4, 200, 128])


torch.Size([4, 200, 128])

# Feed Forward MLP

In [43]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_embd, n_embd * MLP_HIDEN_DIM),
            nn.GELU(),
            nn.Linear(n_embd * MLP_HIDEN_DIM, n_embd)
        )

    def forward(self, x):
        res = self.layers(x)

        return res

# Transformer

In [44]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()

        self.m_heads_attention = MultiHeadAttention(n_embd, n_head)
        self.mlp = FeedForward(n_embd)
        self.heads_norm = nn.LayerNorm(n_embd)
        self.mlp_norm = nn.LayerNorm(n_embd)


    def forward(self, x):
        norm1 = self.heads_norm(x)
        attention = self.m_heads_attention(norm1)
        residual1 = x + attention

        norm2 = self.mlp_norm(residual1)
        mlp = self.mlp(norm2)
        residual2 = residual1 + mlp

        return residual2


# LLM

In [45]:
loss_fn = nn.CrossEntropyLoss()

In [46]:
class LLM(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_TRANSFORMER_BLOCK):
        super().__init__()

        self.emb = Embedding(vocab_size, block_size, n_embd)

        transformer_blocks = [TransformerBlock(n_embd, n_head) for _ in range(n_layer)]
        self.transformer_blocks = nn.Sequential(*transformer_blocks)

        self.norm = nn.LayerNorm(n_embd)
        self.token_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, x, targets=None):
        emb = self.emb(x)

        transformer = self.transformer_blocks(emb)
        norm = self.norm(transformer)

        logits = self.token_layer(norm)

        if (targets is None):
            return logits, None

        loss = loss_fn(logits.reshape((-1, logits.size(dim=-1))), targets.reshape(-1))
        return logits, loss

In [47]:
x = torch.randint(0, 10, (4, 128))
gpt2 = LLM(VOCAB_SIZE, BLOCK_SIZE, N_EMBD, N_HEAD, N_TRANSFORMER_BLOCK)
gpt2(x, torch.randint(0, 10, (4, 128)))[1]

tensor(11.0123, grad_fn=<NllLossBackward0>)

# Training

In [48]:
def slice_train(data):
    x = data[:, :-1]
    y = data[:, 1:]

    return x, y

In [59]:
LEARNING_RATE = 3e-4
EPOCH = 30

In [61]:
gpt2 = LLM(VOCAB_SIZE, BLOCK_SIZE, N_EMBD, N_HEAD, N_TRANSFORMER_BLOCK)
optimizer = torch.optim.AdamW(gpt2.parameters(), lr=LEARNING_RATE)

data = torch.randint(0, VOCAB_SIZE, (4, 129))
x, y = slice_train(data)

In [62]:
for i in range(EPOCH):
    gpt2.train()

    optimizer.zero_grad()
    logits, loss = gpt2(x, y)

    loss.backward()
    optimizer.step()

    if (i % 5 == 0):
        print(loss.item())

11.010024070739746
10.522614479064941
10.03921127319336
9.562792778015137
9.086198806762695
8.60477352142334
